# PFI 2026 · H11-B v2 — Auditoría optimizada de imágenes y máscaras del split axial E9

**Versión corregida para Google Drive:** elimina la indexación recursiva con `os.walk()` que podía demorar mucho tiempo. Resuelve solamente las rutas declaradas en el CSV y las equivalencias de montaje explícitas, con caché y progreso por lotes.

**Solo lectura / CPU:** no entrena, no reevalúa el test, no altera datasets, CSV, modelos ni repositorios. **Si una ruta no puede resolverse unívocamente, deja el control como inconcluso** y solicita agregar una raíz concreta a `EXTRA_DATA_ROOTS`. No asigna máscaras por semejanza de nombres. La lectura SHA-256 de archivos reales puede seguir tardando según tamaño y velocidad de Google Drive.

Conserva la auditoría y salida anonimizada de H11-B. Fuente verificada: partición E9, 610 registros, hash del CSV coincidente con el manifest V2.


In [ ]:
# 1 · Conectar Drive (lectura únicamente)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive/PFI_MVP')
CSV_NAME = 'E9_t2_final_labels_curated_split.csv'
SPLIT_CSV = ROOT / 'results/E9_alkafri_axial_t2_final_labels_baseline' / CSV_NAME
EXPECTED_CSV_SHA256 = '17f78265b93cca59b0680c5fe336943acee78c4fbe34bd86a23df06ce80d051e'

# Solo si imágenes o máscaras están en carpetas distintas: agregar rutas REALES aquí.
EXTRA_DATA_ROOTS = []  # ej.: [Path('/content/drive/MyDrive/OtraCarpeta/AXIAL_ALKAFRI')]

# No efectuar búsqueda recursiva de nombres en Drive. Si hay rutas faltantes,
# indicar raíces precisas en EXTRA_DATA_ROOTS; una coincidencia ambigua nunca se acepta.
ALLOW_BASENAME_SEARCH = False

# Hash de TODOS los archivos accesibles. Puede requerir leer muchos datos desde Drive.
RUN_FULL_HASH = True

# La salida anonimizada se guarda temporalmente bajo /content y se descarga como ZIP.
OUT_DIR = Path('/content/auditoria_pfi_axial_e9_contenidos')

In [ ]:
# 2 · Leer el CSV sin transformarlo y volver a verificar su SHA
import csv, hashlib, json, os, zipfile
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime, timezone
from pathlib import Path

def sha256_path(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

if not SPLIT_CSV.is_file():
    candidates = sorted(ROOT.rglob(CSV_NAME)) if ROOT.exists() else []
    if len(candidates) == 1:
        SPLIT_CSV = candidates[0]
    elif len(candidates) > 1:
        raise RuntimeError(f'Se encontraron {len(candidates)} CSV candidatos. Elegí explícitamente SPLIT_CSV.')
    else:
        raise FileNotFoundError('No encontré el CSV E9. Ajustá SPLIT_CSV en la celda 1.')

csv_sha = sha256_path(SPLIT_CSV)
print('CSV:', SPLIT_CSV.name)
print('SHA-256:', csv_sha)
print('Hash coincide con manifest V2:', csv_sha == EXPECTED_CSV_SHA256)
if csv_sha != EXPECTED_CSV_SHA256:
    raise ValueError('El CSV no coincide con el manifest. No continuar con los hashes de los archivos.')

C_IMAGE, C_MASK, C_PATIENT, C_SPLIT = ('image_file_path', 'final_label_file_path', 'case_id_norm', 'split')
with SPLIT_CSV.open('r', encoding='utf-8-sig', newline='') as f:
    rdr = csv.DictReader(f)
    missing_cols = set((C_IMAGE, C_MASK, C_PATIENT, C_SPLIT)) - set(rdr.fieldnames or [])
    if missing_cols:
        raise ValueError(f'Faltan columnas: {sorted(missing_cols)}')
    rows = []
    for line_no, row in enumerate(rdr, 2):
        values = {key: (row.get(key) or '').strip() for key in (C_IMAGE, C_MASK, C_PATIENT, C_SPLIT)}
        if any(not v or v.lower() == 'nan' for v in values.values()):
            raise ValueError(f'Fila {line_no} tiene datos requeridos vacíos.')
        values[C_SPLIT] = values[C_SPLIT].lower()
        if values[C_SPLIT] not in ('train', 'val', 'test'):
            raise ValueError(f'Split no reconocido en fila {line_no}')
        rows.append(values)

counts = Counter(r[C_SPLIT] for r in rows)
print('Registros por split:', dict(counts), '| total:', len(rows))
print('Pacientes por split:', {s:len({r[C_PATIENT] for r in rows if r[C_SPLIT]==s}) for s in ('train','val','test')})
if dict(counts) != {'train':427,'val':81,'test':102}:
    raise ValueError('Conteos inesperados. Revisar CSV antes de comparar archivos.')

In [ ]:
# 3 · Resolución rápida y estricta de rutas (H11-B v2)
# NO recorrer recursivamente Drive. Probar solo rutas explícitas del CSV,
# traducción de montaje PFI_MVP y bases concretas indicadas por el usuario.
# Si una ruta queda sin resolver o es ambigua, NO se asigna por aproximación.
import os
from functools import lru_cache
from collections import Counter, defaultdict
from pathlib import Path


def normalized_token(token):
    return str(token).replace('\\', '/').strip()

DATA_ROOTS = [
    ROOT / 'data' / 'AXIAL_ALKAFRI',
    ROOT / 'data',
    SPLIT_CSV.parent,
    *[Path(p) for p in EXTRA_DATA_ROOTS],
]
# No recorrer raíces: solo chequear las candidatas exactas requeridas.
DATA_ROOTS = list({str(p): p for p in DATA_ROOTS}.values())
print('Raíces candidatas configuradas:', len(DATA_ROOTS), flush=True)
print('Búsqueda recursiva desactivada; auditando rutas indicadas por el CSV.', flush=True)

@lru_cache(maxsize=30000)
def cached_isfile(path_s):
    try:
        return Path(path_s).is_file()
    except (OSError, ValueError):
        return False

@lru_cache(maxsize=30000)
def resolve_fast(raw_text):
    raw = normalized_token(raw_text)
    p = Path(raw)
    # Caso 1: ruta absoluta vigente en Colab.
    if cached_isfile(str(p)):
        return p, 'exacta'

    # Caso 2: ruta absoluta histórica dentro del workspace PFI_MVP.
    marker = 'PFI_MVP/'
    if marker in raw:
        rel = raw.split(marker, 1)[1]
        moved = ROOT / rel
        if cached_isfile(str(moved)):
            return moved, 'ruta_migrada'

    # Caso 3: datos relativos. Probar únicamente equivalencias explícitas.
    candidates = {}
    suffix = raw.split('AXIAL_ALKAFRI/', 1)[1] if 'AXIAL_ALKAFRI/' in raw else None
    for base in DATA_ROOTS:
        variants = []
        if not p.is_absolute():
            variants.append(base / p)
        if suffix is not None and base.name == 'AXIAL_ALKAFRI':
            variants.append(base / suffix)
        # Basename solo EN EL DIRECTORIO, sin recorrer carpetas de pacientes.
        # No habilitar si hay coincidencias ambiguas en bases diferentes.
        variants.append(base / p.name)
        for candidate in variants:
            cs = str(candidate)
            if cached_isfile(cs):
                candidates[cs] = candidate
    if len(candidates) == 1:
        return next(iter(candidates.values())), 'unica'
    if len(candidates) > 1:
        return None, 'ambigua'
    return None, 'sin_resolver'

resolved = []
resolution_stats = Counter()
print('Comprobando', len(rows), 'registros...', flush=True)
for i, r in enumerate(rows, 1):
    ip, ist = resolve_fast(r[C_IMAGE])
    mp, mst = resolve_fast(r[C_MASK])
    resolution_stats['imagen_' + ist] += 1
    resolution_stats['mascara_' + mst] += 1
    resolved.append({**r, 'image_resolved':ip, 'mask_resolved':mp,
                     'image_status':ist, 'mask_status':mst})
    if i % 50 == 0 or i == len(rows):
        print(f'Resolución: {i}/{len(rows)} registros; '
              f'imágenes no resueltas={resolution_stats["imagen_sin_resolver"]}, '
              f'máscaras no resueltas={resolution_stats["mascara_sin_resolver"]}', flush=True)

unresolved = [r for r in resolved if r['image_resolved'] is None or r['mask_resolved'] is None]
print('Estados de resolución (sin rutas):', dict(resolution_stats), flush=True)
print('Registros con alguna ruta no resuelta/ambigua:', len(unresolved), flush=True)
if unresolved:
    print('INCONCLUSO: faltan rutas explícitas. Configurá EXTRA_DATA_ROOTS con la '
          'carpeta REAL que contiene las imágenes o máscaras y repetí la celda 3.', flush=True)
    print('No concluir ausencia de duplicados con cobertura incompleta.', flush=True)


In [ ]:
# 4 · Auditoría de reuso de rutas imagen-máscara (sin leer píxeles)
# Contabiliza las rutas que son compartidas por múltiples imágenes o pacientes.
# Una máscara repetida es un indicador a revisar; no implica automáticamente error.

paired_resolved = [r for r in resolved if r['image_resolved'] is not None and r['mask_resolved'] is not None]
image_to_mask = defaultdict(set)
mask_to_image = defaultdict(set)
mask_to_patient = defaultdict(set)
mask_to_splits = defaultdict(set)
for r in paired_resolved:
    im, ma, patient, split = (str(r['image_resolved']),str(r['mask_resolved']),r[C_PATIENT],r[C_SPLIT])
    image_to_mask[im].add(ma)
    mask_to_image[ma].add(im)
    mask_to_patient[ma].add(patient)
    mask_to_splits[ma].add(split)

routing = {
    'registros_resueltos_ambos':len(paired_resolved),
    'imagenes_con_multiples_mascaras_por_ruta':sum(len(v)>1 for v in image_to_mask.values()),
    'mascaras_usadas_por_varias_imagenes_por_ruta':sum(len(v)>1 for v in mask_to_image.values()),
    'mascaras_usadas_por_varios_pacientes_por_ruta':sum(len(v)>1 for v in mask_to_patient.values()),
    'mascaras_compartidas_entre_splits_por_ruta':sum(len(v)>1 for v in mask_to_splits.values()),
    'filas_asociadas_a_mascaras_reutilizadas':sum(1 for r in paired_resolved if len(mask_to_image[str(r['mask_resolved'])])>1),
}
print('Relaciones por rutas físicas resueltas:',routing)

In [ ]:
# 5 · Hash SHA-256 del CONTENIDO real (para detectar mismo contenido con nombres distintos)
# Se calcula cada archivo físico una sola vez: lectura intensiva pero sin escrituras.
# En caso de error de lectura, se deja reporte parcial y el control NO puede aprobar.

hash_by_file = {}
hash_errors = []
files_to_hash = set()
for r in resolved:
    for key in ('image_resolved','mask_resolved'):
        if r[key] is not None:
            files_to_hash.add(str(r[key]))

if RUN_FULL_HASH:
    print(f'Archivos físicos distintos a leer: {len(files_to_hash)}')
    for number, path in enumerate(sorted(files_to_hash), 1):
        try:
            hash_by_file[path] = sha256_path(path)
        except Exception as exc:
            hash_errors.append({'tipo':type(exc).__name__})  # no registrar paths privados
        if number % 20 == 0 or number == len(files_to_hash):
            print(f'Progreso: {number}/{len(files_to_hash)} archivos; errores={len(hash_errors)}', flush=True)
else:
    print('Hash completo desactivado; los hallazgos de contenido quedarán SIN VERIFICAR.')

print('Archivos con hash calculado:',len(hash_by_file),'| errores de lectura:',len(hash_errors))

In [ ]:
# 6 · Evaluar duplicados de contenido: imágenes, máscaras y pares
# IMPORTANTE: una máscara vacía repetida puede ser normal. Una imagen idéntica
# ubicada en splits distintos es una alerta mayor que exige investigación.

record_hashes = []
for r in resolved:
    image_path = str(r['image_resolved']) if r['image_resolved'] is not None else None
    mask_path = str(r['mask_resolved']) if r['mask_resolved'] is not None else None
    ihash = hash_by_file.get(image_path)
    mhash = hash_by_file.get(mask_path)
    record_hashes.append({
        'split':r[C_SPLIT], 'patient':r[C_PATIENT],
        'image_hash':ihash, 'mask_hash':mhash,
        'pair_hash':hashlib.sha256((ihash+'|'+mhash).encode('ascii')).hexdigest() if ihash and mhash else None,
    })

def summarize_hash_duplicates(field):
    entries = defaultdict(list)
    for r in record_hashes:
        if r[field] is not None:
            entries[r[field]].append(r)
    duplicates=[rs for rs in entries.values() if len(rs)>1]
    return {
        'grupos_de_contenido_repetido':len(duplicates),
        'registros_en_grupos_repetidos':sum(len(rs) for rs in duplicates),
        'grupos_repetidos_entre_splits':sum(len({r['split'] for r in rs})>1 for rs in duplicates),
        'grupos_repetidos_entre_pacientes':sum(len({r['patient'] for r in rs})>1 for rs in duplicates),
        'grupos_repetidos_dentro_de_un_split':sum(len({r['split'] for r in rs})==1 for rs in duplicates),
    }

content_summary={
    'imagenes':summarize_hash_duplicates('image_hash'),
    'mascaras':summarize_hash_duplicates('mask_hash'),
    'pares_imagen_mascara':summarize_hash_duplicates('pair_hash'),
}

# ¿Cuántas máscaras idénticas por contenido corresponden a diferentes imágenes?
mask_hash_to_images=defaultdict(set)
mask_hash_to_patients=defaultdict(set)
for r in record_hashes:
    if r['mask_hash'] and r['image_hash']:
        mask_hash_to_images[r['mask_hash']].add(r['image_hash'])
        mask_hash_to_patients[r['mask_hash']].add(r['patient'])
content_summary['mascaras']['hashes_que_se_asocian_a_diferentes_imagenes'] = sum(len(v)>1 for v in mask_hash_to_images.values())
content_summary['mascaras']['hashes_que_se_asocian_a_diferentes_pacientes'] = sum(len(v)>1 for v in mask_hash_to_patients.values())

for name, info in content_summary.items():
    print('\n',name, json.dumps(info,ensure_ascii=False,indent=2))

In [ ]:
# 7 · Dictamen técnico y exportación ANONIMIZADA
# No se incluyen IDs, rutas, nombres de estudios, máscaras ni imágenes.
# No declarar un PASS completo si no se resolvió/hashió la totalidad del split.

complete = (len(unresolved) == 0 and len(hash_errors)==0 and
            RUN_FULL_HASH and len(record_hashes) == 610 and
            all(r['image_hash'] and r['mask_hash'] for r in record_hashes))
image_cross = content_summary['imagenes']['grupos_repetidos_entre_splits']
pair_cross = content_summary['pares_imagen_mascara']['grupos_repetidos_entre_splits']
image_cross_patients = content_summary['imagenes']['grupos_repetidos_entre_pacientes']

if not complete:
    verdict='INCONCLUSO_ARCHIVOS_NO_VERIFICADOS'
elif image_cross or pair_cross:
    verdict='ALERTA_DUPLICADOS_DE_IMAGEN_O_PAR_ENTRE_SPLITS'
elif image_cross_patients:
    verdict='REVISAR_IMAGENES_IDENTICAS_ENTRE_PACIENTES'
else:
    verdict='SIN_DUPLICADOS_DE_IMAGEN_ENTRE_SPLITS_POR_CONTENIDO'

summary={
    'hallazgo':'H11-B - contenido y emparejamiento de la particion axial E9',
    'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'csv_sha256':csv_sha,
    'csv_coincide_manifest_v2':csv_sha==EXPECTED_CSV_SHA256,
    'registros_csv':len(rows),
    'particion_registros':dict(counts),
    'resolver_archivos':{'estados':dict(resolution_stats),'filas_no_resueltas':len(unresolved),
                        'filas_resueltas_con_imagen_y_mascara':len(paired_resolved)},
    'hash_contenido':{'activado':RUN_FULL_HASH,'archivos_distintos':len(files_to_hash),
                     'archivos_hasheados':len(hash_by_file),'errores_lectura':len(hash_errors)},
    'relacion_de_rutas':routing,
    'duplicados_por_contenido':content_summary,
    'cobertura_completa':bool(complete),
    'dictamen':verdict,
    'limitaciones':[
        'La igualdad SHA-256 confirma contenido idéntico, no alineación anatómica correcta ni correspondencia semántica de etiquetas.',
        'La repetición de máscaras de contenido (por ejemplo máscaras vacías) NO demuestra leakage por sí sola.',
        'El control de contenidos no reemplaza auditoría de metadatos de paciente, series o geometría de imágenes.',
        'Los conteos de estudios no son independientes si se usa case_id_norm para estudio y paciente.',
        'El test axial E9 se usó previamente en axial-full-v1; no es evidencia de test histórico completamente virgen.',
        'Este notebook no realiza inferencia, entrenamiento, evaluación clínica, ni actualiza metrics/manifest.',
    ]
}

OUT_DIR.mkdir(parents=True,exist_ok=True)
jpath=OUT_DIR/'H11B_reporte_contenidos_anonimizado.json'
mpath=OUT_DIR/'H11B_resumen_para_codex.md'
zip_path=OUT_DIR/'H11B_contenidos_E9_anonimizado.zip'
jpath.write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')

parts=[
    '# H11-B · Evidencia de contenido de imágenes y máscaras (Colab)',
    '',f'- Dictamen: **{verdict}**',
    f'- CSV E9 SHA coincide con manifest: **{summary["csv_coincide_manifest_v2"]}**',
    f'- Archivos únicos leídos/hashiados: **{len(hash_by_file)} / {len(files_to_hash)}**',
    f'- Filas no resueltas: **{len(unresolved)}**; errores de lectura: **{len(hash_errors)}**',
    f'- Rutas de máscaras usadas por varias imágenes: **{routing["mascaras_usadas_por_varias_imagenes_por_ruta"]}**',
    f'- Rutas de máscaras usadas por pacientes distintos: **{routing["mascaras_usadas_por_varios_pacientes_por_ruta"]}**',
    f'- Grupos de imágenes idénticas entre splits: **{image_cross}**',
    f'- Grupos de pares imagen-máscara idénticos entre splits: **{pair_cross}**',
    f'- Grupos de máscaras idénticas entre splits: **{content_summary["mascaras"]["grupos_repetidos_entre_splits"]}**',
    '', '## Lectura técnica',
    ('Se completó el análisis de hashes de los archivos referenciados por E9.' if complete else
     'La cobertura de hashes es parcial o inexistente: NO dar por cerrado el control.'),
    'La repetición de máscaras requiere explicar la relación imagen–máscara; si algunas se comparten por diseño, documentar la justificación con evidencia de origen.',
    '', '## Límites que siguen vigentes',
    *[f'- {x}' for x in summary['limitaciones']],
    '', '## Instrucciones para Codex',
    '- Guardar este informe como evidencia derivada junto con la auditoría H11-A sin copiar rutas ni IDs de pacientes.',
    '- Comprobar el CSV y el snapshot original antes de modificar tablas de tesis.',
    '- Si aparece contenido idéntico entre splits, investigar la procedencia y NO declarar ausencia de leakage.',
    '- NO reinterpretar el test reutilizado como evaluación final externa e independiente.',
    '- No ejecutar entrenamiento, no alterar el baseline axial, las métricas ni los gates para corregir este hallazgo.',
]
mpath.write_text('\n'.join(parts)+'\n',encoding='utf-8')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    archive.write(jpath,jpath.name)
    archive.write(mpath,mpath.name)

print('DICTAMEN:',verdict)
print('Informe JSON:',jpath)
print('Handoff Codex:',mpath)
print('ZIP exportable:',zip_path)
from google.colab import files
files.download(str(zip_path))